# nb11.5 - Lit-Review Network: Finding the Three Strands Your Paper Sits Between

*Week 11, Chapter 11.5 companion. Leah is writing the literature-review paragraph of her
patent-text paper. Her advisor said: "Find the three strands your paper sits between."
Leah has 38 BibTeX entries in `refs.bib` and no idea which three strands are the right ones.
This notebook builds the citation co-occurrence graph from her BibTeX file and clusters it.*

The reveal-the-trick version of the lesson: **the three strands of a literature review are not
chosen by inspiration; they are *found* by looking at which papers cite the same things you do.**
Two papers that share a high fraction of their cited references probably belong to the same
intellectual strand. We will build a graph whose **nodes are your reference list** and whose
**edges are weighted by keyword co-occurrence** (since most BibTeX records do not carry their
*own* reference lists, we cannot do a true citation network; the co-keyword network is the
honest substitute that uses information already in the bib file).

The clustering is then mechanical: find connected components, identify dense subgraphs, name
each subgraph after its highest-degree nodes. The three biggest dense subgraphs are your three
strands. The papers that *bridge* two subgraphs (high betweenness) are the *closest-to-our-work*
citations to highlight in the contribution paragraph.

By the end you will have a hand-rolled BibTeX parser, a co-keyword graph, a connected-components
clusterer, a betweenness-centrality computation, and a one-paragraph draft contribution paragraph
that names three strands plus the bridge papers between them. No networkx required.

In [ ]:
import matplotlib
matplotlib.use("Agg")  # headless: render to buffers, never to a screen

import re
import math
from collections import defaultdict, Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
plt.rcParams["figure.figsize"] = (9, 6)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
pd.set_option("display.float_format", lambda v: f"{v:0.4f}")

## 1. A synthetic BibTeX file as a string

Leah's `refs.bib` is not available offline, so we build a synthetic 30-entry file inline. The
30 entries are clustered by design into three intellectual strands plus a few bridge papers.
The clustering will be recovered from the keyword field; in a real bib file the
`keywords = {...}` line is what carries the signal.

In [ ]:
bib_text = '''@article{bessen2008,
  author = {Bessen, James},
  title = {The value of US patents by owner and patent characteristics},
  journal = {Research Policy},
  year = {2008},
  keywords = {patents, innovation, valuation}
}

@article{hall2005,
  author = {Hall, Bronwyn and Jaffe, Adam and Trajtenberg, Manuel},
  title = {Market value and patent citations},
  journal = {RAND Journal of Economics},
  year = {2005},
  keywords = {patents, citations, innovation}
}

@article{kogan2017,
  author = {Kogan, Leonid and Papanikolaou, Dimitris and Seru, Amit and Stoffman, Noah},
  title = {Technological innovation, resource allocation, and growth},
  journal = {Quarterly Journal of Economics},
  year = {2017},
  keywords = {patents, innovation, growth, asset-pricing}
}

@article{trajtenberg1990,
  author = {Trajtenberg, Manuel},
  title = {A penny for your quotes: patent citations and the value of innovations},
  journal = {RAND Journal of Economics},
  year = {1990},
  keywords = {patents, citations, innovation}
}

@article{jaffe1989,
  author = {Jaffe, Adam},
  title = {Real effects of academic research},
  journal = {American Economic Review},
  year = {1989},
  keywords = {patents, spillovers, innovation}
}

@article{bloom2013,
  author = {Bloom, Nicholas and Schankerman, Mark and Van Reenen, John},
  title = {Identifying technology spillovers and product market rivalry},
  journal = {Econometrica},
  year = {2013},
  keywords = {patents, spillovers, productivity}
}

@article{moser2013,
  author = {Moser, Petra},
  title = {Patents and innovation: evidence from economic history},
  journal = {Journal of Economic Perspectives},
  year = {2013},
  keywords = {patents, innovation, history}
}

@article{lerner2002,
  author = {Lerner, Josh},
  title = {Patent policy innovations: a clinical examination},
  journal = {Journal of Economic Perspectives},
  year = {2002},
  keywords = {patents, policy, innovation}
}

@article{argente2020,
  author = {Argente, David and Lee, Munseob and Moreira, Sara},
  title = {Innovation, patents, and product creation},
  journal = {Journal of Political Economy},
  year = {2020},
  keywords = {patents, innovation, product-creation}
}

@article{hsu2014,
  author = {Hsu, David and Ziedonis, Rosemarie},
  title = {Resources as dual sources of advantage in patent races},
  journal = {Strategic Management Journal},
  year = {2014},
  keywords = {patents, strategy, innovation}
}

@article{tetlock2007,
  author = {Tetlock, Paul},
  title = {Giving content to investor sentiment: the role of media in the stock market},
  journal = {Journal of Finance},
  year = {2007},
  keywords = {text-analysis, sentiment, asset-pricing}
}

@article{loughran2011,
  author = {Loughran, Tim and McDonald, Bill},
  title = {When is a liability not a liability? Textual analysis, dictionaries, and 10-Ks},
  journal = {Journal of Finance},
  year = {2011},
  keywords = {text-analysis, sentiment, 10-K, disclosures}
}

@article{hoberg2010,
  author = {Hoberg, Gerard and Phillips, Gordon},
  title = {Product market synergies and competition in mergers and acquisitions},
  journal = {Review of Financial Studies},
  year = {2010},
  keywords = {text-analysis, product-similarity, mergers}
}

@article{lopez2017,
  author = {Lopez-Lira, Alejandro},
  title = {Risk factors that matter: textual analysis of risk disclosures},
  journal = {Review of Financial Studies},
  year = {2017},
  keywords = {text-analysis, risk-disclosures, 10-K}
}

@article{cohen2013,
  author = {Cohen, Lauren and Diether, Karl and Malloy, Christopher},
  title = {Misvaluing innovation},
  journal = {Review of Financial Studies},
  year = {2013},
  keywords = {text-analysis, innovation, asset-pricing}
}

@article{packalen2015,
  author = {Packalen, Mikko and Bhattacharya, Jay},
  title = {Cities and ideas},
  journal = {NBER Working Paper},
  year = {2015},
  keywords = {patents, text-analysis, innovation}
}

@article{aghion2005,
  author = {Aghion, Philippe and Bloom, Nicholas and Blundell, Richard and Griffith, Rachel and Howitt, Peter},
  title = {Competition and innovation: an inverted-U relationship},
  journal = {Quarterly Journal of Economics},
  year = {2005},
  keywords = {innovation, competition, growth}
}

@article{romer1990,
  author = {Romer, Paul},
  title = {Endogenous technological change},
  journal = {Journal of Political Economy},
  year = {1990},
  keywords = {innovation, growth, theory}
}

@article{acemoglu2018,
  author = {Acemoglu, Daron and Akcigit, Ufuk and Alp, Harun and Bloom, Nicholas and Kerr, William},
  title = {Innovation, reallocation, and growth},
  journal = {American Economic Review},
  year = {2018},
  keywords = {innovation, growth, productivity}
}

@article{akcigit2016,
  author = {Akcigit, Ufuk and Hanley, Douglas and Serrano-Velarde, Nicolas},
  title = {Back to basics: basic research spillovers},
  journal = {Review of Economic Studies},
  year = {2016},
  keywords = {innovation, growth, spillovers}
}

@article{aghion2014,
  author = {Aghion, Philippe and Akcigit, Ufuk and Howitt, Peter},
  title = {What do we learn from Schumpeterian growth theory?},
  journal = {Handbook of Economic Growth},
  year = {2014},
  keywords = {innovation, growth, theory}
}

@article{bloom2020,
  author = {Bloom, Nicholas and Jones, Charles and Van Reenen, John and Webb, Michael},
  title = {Are ideas getting harder to find?},
  journal = {American Economic Review},
  year = {2020},
  keywords = {innovation, growth, productivity}
}

@article{antras2017,
  author = {Antras, Pol and Yeaple, Stephen},
  title = {Multinational firms and the structure of international trade},
  journal = {Handbook of International Economics},
  year = {2017},
  keywords = {innovation, trade, multinational}
}

@article{kortum1999,
  author = {Kortum, Samuel and Lerner, Josh},
  title = {What is behind the recent surge in patenting?},
  journal = {Research Policy},
  year = {1999},
  keywords = {patents, policy, innovation}
}

@article{hall2001,
  author = {Hall, Bronwyn and Jaffe, Adam and Trajtenberg, Manuel},
  title = {The NBER patent citation data file: lessons, insights, and methodological tools},
  journal = {NBER Working Paper},
  year = {2001},
  keywords = {patents, citations, data}
}

@article{lanjouw2004,
  author = {Lanjouw, Jean and Schankerman, Mark},
  title = {Patent quality and research productivity: measuring innovation with multiple indicators},
  journal = {Economic Journal},
  year = {2004},
  keywords = {patents, innovation, indicators}
}

@article{wuchty2007,
  author = {Wuchty, Stefan and Jones, Benjamin and Uzzi, Brian},
  title = {The increasing dominance of teams in production of knowledge},
  journal = {Science},
  year = {2007},
  keywords = {innovation, teams, science}
}

@article{benner2008,
  author = {Benner, Mary and Waldfogel, Joel},
  title = {Close to you? Bias and precision in patent-based measures},
  journal = {Research Policy},
  year = {2008},
  keywords = {patents, measurement, innovation}
}

@article{griliches1990,
  author = {Griliches, Zvi},
  title = {Patent statistics as economic indicators: a survey},
  journal = {Journal of Economic Literature},
  year = {1990},
  keywords = {patents, indicators, innovation}
}

@article{kelly2021,
  author = {Kelly, Bryan and Papanikolaou, Dimitris and Seru, Amit and Taddy, Matt},
  title = {Measuring technological innovation over the long run},
  journal = {American Economic Review: Insights},
  year = {2021},
  keywords = {patents, text-analysis, innovation}
}
'''
print(f"bib_text length = {len(bib_text)} chars")


## 2. Hand-rolling a BibTeX parser (no bibtexparser needed)

We avoid the bibtexparser dependency to keep the pinned stack minimal. The BibTeX format is
forgiving but predictable: each entry starts with `@TYPE{key,`, then a sequence of
`field = {value},` lines, then a closing `}`. Our parser handles single-line values (the
common case) and ignores everything fancier. It returns a list of dicts.

In [ ]:
ENTRY_HEAD_RE = re.compile(r"@(\w+)\s*\{\s*([A-Za-z0-9_:\-]+)\s*,")
FIELD_RE = re.compile(r"^\s*(\w+)\s*=\s*\{(.*?)\}\s*,?\s*$")

def parse_bib(text):
    entries = []
    blocks = re.split(r"\n(?=@\w+\{)", text.strip())
    for block in blocks:
        head_match = ENTRY_HEAD_RE.match(block)
        if not head_match:
            continue
        entry_type = head_match.group(1).lower()
        key = head_match.group(2)
        fields = {}
        body = block[head_match.end():]
        for line in body.splitlines():
            m = FIELD_RE.match(line)
            if m:
                fname = m.group(1).lower()
                fval = m.group(2).strip()
                fields[fname] = fval
        entry = {"key": key, "type": entry_type, **fields}
        entries.append(entry)
    return entries

entries = parse_bib(bib_text)
print(f"Parsed {len(entries)} entries.")
df_entries = pd.DataFrame(entries)
df_entries.head()

**A sanity check.** Thirty entries parsed; the dataframe shows the `key`, `author`, `title`,
`year`, `keywords` columns. The `keywords` column is the one the network will use - the
comma-separated tokens are the *intellectual fingerprint* of each paper.

## 3. Building the co-keyword adjacency matrix

For each pair of entries $(i, j)$ the **weight** of the edge is the Jaccard similarity of their
keyword sets: $w_{ij} = |K_i \cap K_j| / |K_i \cup K_j|$. Two papers with identical keyword
sets get $w = 1$; two with no overlap get $w = 0$. We threshold at $w \ge 0.30$ to drop edges
that are too weak to mean anything.

In [ ]:
def keywords_set(entry):
    raw = entry.get("keywords", "")
    return frozenset(k.strip().lower() for k in raw.split(",") if k.strip())

def jaccard(a, b):
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)

n = len(entries)
W = np.zeros((n, n))
keys = [keywords_set(e) for e in entries]
for i in range(n):
    for j in range(i + 1, n):
        w = jaccard(keys[i], keys[j])
        W[i, j] = W[j, i] = w

THRESHOLD = 0.30
A = (W >= THRESHOLD).astype(int)
np.fill_diagonal(A, 0)
print(f"Built {n}x{n} adjacency. Edges (above threshold {THRESHOLD}): {int(A.sum() // 2)}")
print(f"Average degree: {A.sum(axis=1).mean():0.2f}")

## 4. Visualizing the matrix as a heatmap (the "who shares what" view)

A reordered adjacency heatmap is the easiest first look at the structure. If our entries
really cluster into strands, the heatmap will show *block-diagonal density*: rectangles of
dark cells along the diagonal corresponding to within-strand connections. We will leave the
node order alone for now and reorder it after clustering in section 6.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
labels = [e["key"] for e in entries]
im = ax.imshow(A, cmap="Greys", aspect="auto")
ax.set_xticks(range(n)); ax.set_xticklabels(labels, rotation=90, fontsize=6)
ax.set_yticks(range(n)); ax.set_yticklabels(labels, fontsize=6)
ax.set_title("Co-keyword adjacency (Jaccard >= 0.30): unordered")
plt.tight_layout(); plt.close(fig)
print("heatmap rendered (unordered)")

## 5. Connected components: a quick first pass at clustering

Before any fancy modularity-maximization, we run the cheap algorithm: union-find on the binary
adjacency matrix. The result tells us how many *connected* islands the graph has. If the answer
is one giant component plus a few isolated singletons, the next step (community detection) is
worth running. If the answer is several balanced components, the connected components are
already the strands.

In [ ]:
class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))
    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.parent[ra] = rb

uf = UnionFind(n)
for i in range(n):
    for j in range(i + 1, n):
        if A[i, j]:
            uf.union(i, j)
roots = [uf.find(i) for i in range(n)]
component_of = {}
for i, r in enumerate(roots):
    component_of[i] = r
size_of_root = Counter(roots)
print(f"Connected components: {len(set(roots))}")
print(f"Largest components: {size_of_root.most_common(5)}")

**Reading the components.** If the threshold of 0.30 is too generous, you get one giant
component. If it is too strict, you get many singletons. Our 30-entry bib is *engineered* to
sit in the middle: a handful of meaningful components plus a couple of isolated nodes. The
sizes tell us roughly which strands are large enough to name.

## 6. Modularity-based community detection: the Louvain-greedy hand roll

The connected-components view is too coarse. Within a single connected component there can
still be distinct strands. The classical solution is **Louvain community detection**, but the
full Louvain algorithm is heavy and out of scope. We will hand-roll a *greedy modularity*
optimizer: start with every node in its own community, then iteratively merge the two
communities whose merge increases modularity the most, stopping when no merge improves it.
This is the Clauset-Newman-Moore algorithm in its simplest form, and it is sufficient when
the graph has at most a few hundred nodes.

In [ ]:
def modularity(A, communities):
    m = A.sum() / 2.0
    if m == 0:
        return 0.0
    deg = A.sum(axis=1)
    Q = 0.0
    for community in communities:
        for i in community:
            for j in community:
                Q += A[i, j] - deg[i] * deg[j] / (2 * m)
    return Q / (2 * m)

def greedy_modularity(A, max_iter=200):
    n = A.shape[0]
    communities = [{i} for i in range(n)]
    best_Q = modularity(A, communities)
    for _ in range(max_iter):
        best_merge = None
        best_gain = 0.0
        for a in range(len(communities)):
            for b in range(a + 1, len(communities)):
                # Heuristic: only consider merging if there is any edge between the communities.
                ca, cb = communities[a], communities[b]
                has_edge = False
                for i in ca:
                    for j in cb:
                        if A[i, j]:
                            has_edge = True; break
                    if has_edge: break
                if not has_edge: continue
                merged = communities[:a] + [ca | cb] + communities[a + 1:b] + communities[b + 1:]
                Q_new = modularity(A, merged)
                gain = Q_new - best_Q
                if gain > best_gain:
                    best_gain = gain
                    best_merge = (a, b)
        if best_merge is None:
            break
        a, b = best_merge
        communities = communities[:a] + [communities[a] | communities[b]] + communities[a + 1:b] + communities[b + 1:]
        best_Q += best_gain
    return communities, best_Q

communities, Q = greedy_modularity(A)
sizes = sorted([len(c) for c in communities], reverse=True)
print(f"Found {len(communities)} communities; sizes (top 8): {sizes[:8]}")
print(f"Modularity Q = {Q:0.4f}")

**Reading the modularity output.** A Q value above 0.3 is the usual rule of thumb for a
graph with meaningful community structure. Below 0.3 is "random graph" territory; above 0.6
is "highly modular." Our synthetic bib should land in the 0.4-0.6 range because the strands
are real (three keyword clusters drove the construction).

## 7. Naming the three strands: keywords-of-the-community

For each community we tabulate which keywords are most concentrated in its members. The
top three keywords name the strand. This step is what your contribution paragraph reuses
verbatim.

In [ ]:
def strand_keywords(community, entries, top_k=4):
    bag = Counter()
    for i in community:
        for kw in keywords_set(entries[i]):
            bag[kw] += 1
    return bag.most_common(top_k)

# Sort communities by size, descending
communities_sorted = sorted(communities, key=len, reverse=True)
top_strands = []
for c_idx, comm in enumerate(communities_sorted[:5]):
    kws = strand_keywords(comm, entries)
    members = [entries[i]["key"] for i in comm]
    top_strands.append((c_idx, len(comm), kws, members))
    print(f"\nStrand {c_idx + 1} ({len(comm)} papers)")
    print(f"  top keywords: {kws}")
    print(f"  members ({min(8, len(members))} shown): {members[:8]}")

**Reading the strands.** The top three (or top four, depending on graph) should map to:
*patents* (Bessen, Hall-Jaffe-Trajtenberg, Lerner, Kortum, Argente, ...); *text-analysis*
(Tetlock, Loughran-McDonald, Hoberg-Phillips, Lopez-Lira, ...); and *innovation-and-growth*
(Aghion, Romer, Acemoglu, Bloom-Jones-Van-Reenen-Webb, ...). The *bridge* papers - Kelly et al.
(2021) which uses text analysis on patents, or Cohen et al. (2013) which uses text on
innovation - should be the ones the betweenness step picks up.

## 8. Betweenness centrality: finding the bridge papers

Betweenness asks: how many *shortest paths* between any two other nodes pass through this
node? A node with high betweenness sits *between* strands - which is exactly what the
"closest-to-our-work" papers do in a contribution paragraph. We compute betweenness via
unweighted BFS (Brandes 2001 algorithm, simplified).

In [ ]:
def betweenness_unweighted(A):
    n = A.shape[0]
    bc = np.zeros(n)
    for s in range(n):
        # BFS from s
        S = []
        P = [[] for _ in range(n)]
        sigma = np.zeros(n); sigma[s] = 1
        dist = -np.ones(n, dtype=int); dist[s] = 0
        Q = [s]
        while Q:
            v = Q.pop(0)
            S.append(v)
            for w in range(n):
                if A[v, w] == 0:
                    continue
                if dist[w] < 0:
                    dist[w] = dist[v] + 1
                    Q.append(w)
                if dist[w] == dist[v] + 1:
                    sigma[w] += sigma[v]
                    P[w].append(v)
        delta = np.zeros(n)
        while S:
            w = S.pop()
            for v in P[w]:
                delta[v] += (sigma[v] / sigma[w]) * (1 + delta[w])
            if w != s:
                bc[w] += delta[w]
    return bc / 2.0   # undirected graph correction

bc = betweenness_unweighted(A)
node_table = pd.DataFrame({
    "key": [e["key"] for e in entries],
    "year": [int(e.get("year", 0)) for e in entries],
    "degree": A.sum(axis=1),
    "betweenness": bc,
}).sort_values("betweenness", ascending=False)
print("\nTop-10 by betweenness (the bridge papers):")
print(node_table.head(10).to_string(index=False))

**Reading the table.** The papers with the highest betweenness are the *closest-to-our-
work* candidates. Kelly et al. (2021), Packalen and Bhattacharya (2015), and Cohen, Diether
and Malloy (2013) typically rise to the top because they all sit at the intersection of two
strands (patents+text-analysis; patents+innovation; text-analysis+innovation). These are the
papers Leah will name in the contribution paragraph.

## 9. A node-link plot: laying the graph out on the page

We hand-roll a Fruchterman-Reingold spring layout - again, no networkx. Nodes repel each other
with force $1/d^2$, edges contract with force $d$. We iterate for 100 steps with a cooling
schedule. The result is a visually informative embedding of the network.

In [ ]:
def spring_layout(A, iters=120, seed=42):
    n = A.shape[0]
    layout_rng = np.random.default_rng(seed)
    pos = layout_rng.uniform(-1, 1, size=(n, 2))
    k = math.sqrt(1.0 / n)
    for step in range(iters):
        t = 0.1 * (1 - step / iters)        # cooling temperature
        disp = np.zeros_like(pos)
        # Repulsion
        for i in range(n):
            delta = pos[i] - pos
            dist = np.linalg.norm(delta, axis=1)
            dist[dist == 0] = 1e-3
            f = (k * k) / dist[:, None]
            disp[i] += (delta * f).sum(axis=0)
        # Attraction along edges
        for i in range(n):
            for j in range(n):
                if A[i, j] == 0: continue
                delta = pos[i] - pos[j]
                d = max(np.linalg.norm(delta), 1e-3)
                f = (d * d) / k
                disp[i] -= delta * (f / d)
        # Update with cooling
        length = np.linalg.norm(disp, axis=1)
        length[length == 0] = 1e-3
        pos += (disp.T * (np.minimum(length, t) / length)).T
        pos = np.clip(pos, -1.5, 1.5)
    return pos

pos = spring_layout(A, iters=100)
print(f"layout: {pos.shape}, range x: ({pos[:,0].min():0.2f}, {pos[:,0].max():0.2f})")

In [ ]:
community_color = {}
palette = ["#27ae60", "#2980b9", "#c0392b", "#8e44ad", "#f39c12", "#16a085", "#7f8c8d"]
for c_idx, comm in enumerate(communities_sorted):
    for i in comm:
        community_color[i] = palette[c_idx % len(palette)]

fig, ax = plt.subplots(figsize=(10, 8))
# Edges first
for i in range(n):
    for j in range(i + 1, n):
        if A[i, j]:
            ax.plot([pos[i, 0], pos[j, 0]], [pos[i, 1], pos[j, 1]],
                    color="#cccccc", linewidth=0.5, zorder=1)
# Nodes sized by degree
deg = A.sum(axis=1)
sizes = 50 + 30 * deg
colors = [community_color[i] for i in range(n)]
ax.scatter(pos[:, 0], pos[:, 1], s=sizes, c=colors, edgecolors="black",
           linewidths=0.5, zorder=2)
# Label top-betweenness nodes
top_bc = np.argsort(bc)[-6:]
for i in top_bc:
    ax.annotate(entries[i]["key"], (pos[i, 0], pos[i, 1]),
                fontsize=8, alpha=0.85, ha="center")
ax.set_title("Co-keyword network of Leah's bib (top-6 bridge papers labeled)")
ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.close(fig)
print("network plot rendered")

**Reading the plot.** Three dense blobs in three colors should be visible, with thin
ligaments connecting them through the high-betweenness bridge nodes. The bridge labels are
the citations you call out in the contribution paragraph. If your real bib does *not* show
this pattern, the diagnosis is either (a) your keywords are too noisy (and need cleaning) or
(b) your paper has only one strand, not three (and you may not yet be writing the paper you
think you are).

## 10. The auto-generated contribution paragraph

Given the three strands and the bridge papers, we can mechanically draft the first version of
the contribution paragraph. This is intended as a *scaffold* - you will rewrite every
sentence - but the scaffold tells you what to mention.

In [ ]:
def name_strand(kws):
    # Drop generic words; pick the most informative
    generic = {"innovation"}
    informative = [k for k, _ in kws if k not in generic]
    return " + ".join(informative[:2]) if informative else kws[0][0]

def auto_contribution(strands_info, entries, bc_arr, top_bridges=3):
    lines = []
    lines.append("Our paper contributes to three strands of the literature.")
    for idx, (c_idx, n_papers, kws, members) in enumerate(strands_info[:3]):
        strand_name = name_strand(kws)
        examples = ", ".join(members[:3])
        verbs = ["extend that work by", "differ from that work by", "depart from that work by"]
        lines.append(f"The {['first','second','third'][idx]} strand is the {strand_name} "
                     f"literature, exemplified by {examples} ({n_papers} papers in our bibliography). "
                     f"We {verbs[idx]} <fill in: what is your specific innovation>.")
    bridges_idx = np.argsort(bc_arr)[-top_bridges:][::-1]
    bridge_keys = [entries[i]["key"] for i in bridges_idx]
    lines.append(f"Closest to our work are {', '.join(bridge_keys)}, which sit at the "
                 f"intersection of these strands and motivate our particular question.")
    return " ".join(lines)

draft_paragraph = auto_contribution(top_strands, entries, bc, top_bridges=3)
print(draft_paragraph)

**Reading the auto-draft.** This is not a finished contribution paragraph - it has
`<fill in:>` placeholders for the *what makes your paper different* clauses, which only you
can write. But the **structure** is right: three strands named, each illustrated with three
papers, then the bridge papers identified by name. This skeleton is what your real
contribution paragraph will look like once you fill the placeholders.

## 11. When the network analysis fails (be honest)

Three failure modes worth knowing.

1. **Garbage keywords in, garbage clusters out.** If your bib's keyword fields are
   inconsistent (`patents` vs `Patents`; `text-analysis` vs `textual analysis` vs `NLP`), the
   Jaccard similarity will underestimate true overlap. Run a normalization pass first: lower
   case, strip plurals, collapse synonyms.

2. **Threshold sensitivity.** A Jaccard threshold of 0.30 is conservative; 0.20 admits more
   edges, 0.40 fewer. Run the analysis at three thresholds and report the strands that are
   robust across all three.

3. **Bib does not have keywords.** Many JF/RFS/JFE BibTeX entries do not include the
   `keywords =` field. In that case, mine keywords from titles and abstracts. We do not do
   that here; the principle generalizes.

## 12. Your turn

1. **Threshold sweep.** Re-run the modularity computation at thresholds 0.20, 0.30, 0.40,
   0.50. Plot the number of communities and the modularity Q as a function of threshold.
   Pick the threshold that gives the most stable structure.

2. **Add ten more entries.** Insert ten more BibTeX entries to the bib_text string. Verify
   that the three strands the network finds remain stable as the bib grows. (If they shift
   dramatically, you have a graph-stability problem worth reporting in the manuscript.)

3. **Replace the BibTeX parser.** Install `bibtexparser` and check that our hand-rolled parser
   agrees with it on the synthetic bib. Then add a third backend that extracts keywords from
   `title +abstract` when the `keywords` field is missing.

4. **Real bib.** Export your own paper's references as BibTeX, normalize keywords, and run
   the whole pipeline on your real data. The three strands the network finds should match
   the three strands you would have named manually. If they differ, that is information.

**Citations to anchor your writeup.** Newman (2004, *Phys Rev E* 69:066133) on modularity;
Clauset, Newman, Moore (2004, *Phys Rev E* 70:066111) on greedy modularity optimization;
Brandes (2001, *J Math Sociol* 25(2):163-177) on betweenness centrality; Hall, Jaffe and
Trajtenberg (2001, NBER WP) on the patent-citation graph that inspired this entire literature.